In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Labs"
)

xpt_file = DATA_DIR / "GHB_F.xpt"
csv_file = DATA_DIR / "GHB_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched source values.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 GHB_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBXGH"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "GHB_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 7369:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 2:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 7,369 rows x 2 columns."
)


# ============================================================
# 4. CHECK FOR KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# GHB_F should NOT contain this artifact.
#
# Do not replace anything automatically if it appears.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "were found in GHB_F. "
        "Do not automatically replace them."
    )


print(
    "PASS: No tiny XPORT zero-decoding artifact exists."
)


# ============================================================
# 5. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# LBXGH is a genuine continuous laboratory measurement.
#
# Values such as:
#
#     5.2
#     5.7
#     6.0
#     16.4
#
# are legitimate.
#
# DO NOT:
#
#     round LBXGH
#     convert LBXGH to Int64
#     remove legitimate decimal representation
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )


        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "genuine fractional values"
        )


# ============================================================
# 6. VALIDATE EXPECTED DECIMAL VARIABLES
# ============================================================

DECIMAL_COLUMNS = [
    "LBXGH"
]


if fractional_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern differs "
        "from validated GHB_F pattern.\n"
        f"Expected: {DECIMAL_COLUMNS}\n"
        f"Actual:   {fractional_columns}"
    )


print(
    "\nPASS: LBXGH correctly identified "
    "as a genuine decimal measurement."
)


print(
    "PASS: LBXGH will NOT be rounded "
    "or converted to Int64."
)


# ============================================================
# 7. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 8. PRESERVE LBXGH AS DECIMAL
# ============================================================

#
# IMPORTANT:
#
# Do NOT convert LBXGH to Int64.
#
# Even values that happen to be mathematically whole,
# such as:
#
#     6.0
#
# belong to a continuous laboratory measurement
# variable and should remain in the decimal column.
#

df["LBXGH"] = (
    df["LBXGH"]
    .astype(float)
)


print(
    "\nPASS: LBXGH retained as decimal measurement."
)


# ============================================================
# 9. CDC CHECK - LBXGH
# ============================================================

#
# CDC:
#
# Glycohemoglobin (%)
#
# Range:
#     4.0 to 16.4
#
# Non-missing:
#     6,930
#
# Missing:
#     439
#

s = df["LBXGH"]


print("\n--- LBXGH ---")


print(
    "Minimum:",
    s.min()
)


print(
    "Maximum:",
    s.max()
)


print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not np.isclose(
    s.min(),
    4.0,
    rtol=0,
    atol=0
):

    raise ValueError(
        "LBXGH minimum differs from CDC."
    )


if not np.isclose(
    s.max(),
    16.4,
    rtol=0,
    atol=0
):

    raise ValueError(
        "LBXGH maximum differs from CDC."
    )


if int(
    s.notna()
    .sum()
) != 6930:

    raise ValueError(
        "LBXGH non-missing count differs from CDC."
    )


if int(
    s.isna()
    .sum()
) != 439:

    raise ValueError(
        "LBXGH missing count differs from CDC."
    )


print(
    "PASS: LBXGH matches CDC."
)


# ============================================================
# 10. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# No zero restoration or numeric correction is needed
# in GHB_F.
#
# Therefore:
#
# SEQN numeric value must equal original SEQN.
# LBXGH must equal original LBXGH exactly.
#
# Only SEQN display changes:
#
#     51624.0 -> 51624
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


#
# SEQN
#
expected_seqn = (
    source_df["SEQN"]
    .to_numpy()
)


current_seqn = (
    df["SEQN"]
    .astype("Float64")
    .to_numpy(
        dtype=float,
        na_value=np.nan
    )
)


if not np.allclose(
    expected_seqn,
    current_seqn,
    rtol=0,
    atol=0,
    equal_nan=True
):

    failures.append(
        "SEQN"
    )


#
# LBXGH
#
expected_lbxgh = (
    source_df["LBXGH"]
    .to_numpy()
)


current_lbxgh = (
    df["LBXGH"]
    .to_numpy(
        dtype=float
    )
)


if not np.allclose(
    expected_lbxgh,
    current_lbxgh,
    rtol=0,
    atol=0,
    equal_nan=True
):

    failures.append(
        "LBXGH"
    )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: Both variables preserve "
    "original XPT numeric values exactly."
)


# ============================================================
# 11. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format.
# NO rounding.
#
# SEQN is Int64.
# LBXGH remains float.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 12. RE-READ EXPORTED CSV
# ============================================================

#
# float_precision="round_trip" is used so
# LBXGH can be compared exactly after export.
#

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 13. EXACT XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: Both CSV columns match "
    "source XPT numeric values exactly."
)


# ============================================================
# 14. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 15. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Unexpected tiny XPORT artifact found "
        "in final CSV."
    )


print(
    "PASS: No tiny-value artifacts."
)


# ============================================================
# 16. SEQN .0 FORMAT CHECK
# ============================================================

#
# IMPORTANT:
#
# Do NOT globally search the entire CSV for ".0".
#
# LBXGH is a continuous decimal variable and values
# such as 6.0 are valid.
#
# Only SEQN must not end in ".0".
#

print("\n--- SEQN FORMAT CHECK ---")


seqn_dot_zero_errors = []


with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(
        f
    )


    for row_number, row in enumerate(
        reader,
        start=2
    ):

        value = row["SEQN"]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            seqn_dot_zero_errors.append(
                (
                    row_number,
                    value
                )
            )


            if len(
                seqn_dot_zero_errors
            ) >= 20:

                break


print(
    "SEQN '.0' formatting errors:",
    len(
        seqn_dot_zero_errors
    )
)


if seqn_dot_zero_errors:

    print(
        seqn_dot_zero_errors
    )

    raise ValueError(
        "Unexpected .0 formatting remains in SEQN."
    )


print(
    "PASS: SEQN has no unnecessary .0."
)


# ============================================================
# 17. EXACT LBXGH DECIMAL FIDELITY
# ============================================================

#
# Explicitly verify HbA1c values after CSV export.
#
# This is the most important measurement fidelity check.
#

print("\n--- LBXGH DECIMAL FIDELITY ---")


expected_values = (
    source_df["LBXGH"]
    .to_numpy()
)


exported_values = (
    csv_df["LBXGH"]
    .to_numpy()
)


if not np.allclose(
    expected_values,
    exported_values,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "LBXGH values changed during CSV export."
    )


print(
    "PASS: LBXGH values preserved exactly."
)


# ============================================================
# 18. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print("\n--- SCIENTIFIC NOTATION CHECK ---")


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 19. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 20. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "GHB_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Tiny-value corrections: NONE"
)


print(
    "Continuous decimal variables preserved: LBXGH"
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0 in SEQN: 0 expected"
)


print(
    "LBXGH decimal values changed: 0 expected"
)

NHANES 2009-2010 GHB_F XPT -> CSV
Rows: 7,369
Columns: 2
PASS: Structure = 7,369 rows x 2 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 0
PASS: No tiny XPORT zero-decoding artifact exists.

--- FRACTIONAL VALUE CHECK ---
LBXGH: 6,302 genuine fractional values

PASS: LBXGH correctly identified as a genuine decimal measurement.
PASS: LBXGH will NOT be rounded or converted to Int64.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51624
Maximum: 62160
PASS: SEQN validated.

PASS: LBXGH retained as decimal measurement.

--- LBXGH ---
Minimum: 4.0
Maximum: 16.4
Non-missing: 6930
Missing: 439
PASS: LBXGH matches CDC.

--- VALUE FIDELITY BEFORE EXPORT ---
PASS: Both variables preserve original XPT numeric values exactly.

CSV created:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_Predictor\data\2009_2010\Labs\GHB_F_corrected.csv

--- XPT -> CSV FIDELITY ---
PASS: Both CSV columns match source 